In [1]:
import re
import json
import time
import random
import torch
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm
from transformers import pipeline

# =========================================================
# CONFIG
# =========================================================

MAX_PAGES_PER_CATEGORY = 8
TARGET_NUM_ARTICLES = 500

MAX_INPUT_CHARS = 2500
BATCH_SIZE_QA = 4

OUTPUT_FILE = "uet_qa_dataset.json"

# =========================================================
# HELPERS
# =========================================================

def parse_qa_to_dataframe(docs):
    """
    Bóc tách CH/ĐA từ generated QA để xuất CSV.
    Hỗ trợ xử lý các biến thể markdown như **CH1**:, CH 1:, ĐA 1: từ LLM.
    """
    qa_list = []

    for doc in docs:
        qa_text = doc.get("generated_qa", "")

        # Regex nâng cấp:
        # - Cho phép có hoặc không có dấu bôi đậm (**)
        # - Cho phép có hoặc không có khoảng trắng giữa chữ và số (CH1 hoặc CH 1)
        # - Cho phép dấu hai chấm viết liền hoặc cách ra (: hoặc  :)
        pairs = re.findall(
            r"(?:\*\*|)?CH\s*\d+\s*(?:\*\*|)?\s*:\s*(.*?)\s*\n(?:\*\*|)?ĐA\s*\d+\s*(?:\*\*|)?\s*:\s*(.*?)(?=\n(?:\*\*|)?CH\s*\d+:|\n(?:\*\*|)?CH\s*\d+\s*(?:\*\*|)?\s*:|$)",
            qa_text + "\n",
            re.DOTALL | re.IGNORECASE
        )

        for q, a in pairs:
            q = q.strip()
            a = a.strip()

            # Loại bỏ các ký tự markdown thừa nếu mô hình còn sót lại
            q = re.sub(r"^\*+\s*|\*+\s*$", "", q)
            a = re.sub(r"^\*+\s*|\*+\s*$", "", a)

            if q and a:
                qa_list.append({
                    "Title": doc.get("title"),
                    "URL": doc.get("url"),
                    "Question": q,
                    "Answer": a,
                    "Context": doc.get("rule_cleaned_content") # Nên lưu thêm trường này để làm Ground Truth Context sau này
                })

    return pd.DataFrame(qa_list)
# =========================================================
# LOAD MODEL
# =========================================================

print("\n====================")
print("LOADING MODEL")
print("====================")

qa_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    model_kwargs={
        "attn_implementation": "sdpa"
    }
)

qa_pipe.tokenizer.padding_side = "left"
qa_pipe.tokenizer.pad_token_id = qa_pipe.tokenizer.eos_token_id

# =========================================================
# STEP 1: GET LINKS
# =========================================================

def get_links_from_category(category_url, max_pages=2):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    post_links = []

    print(f"\nScanning category: {category_url}")

    for page in range(1, max_pages + 1):

        current_url = (
            category_url
            if page == 1
            else f"{category_url.rstrip('/')}/page/{page}/"
        )

        print(f" -> Page {page}")

        try:

            response = requests.get(
                current_url,
                headers=headers,
                timeout=15
            )

            if response.status_code != 200:
                print(f"Stop scanning at page {page}: {response.status_code}")
                break

            soup = BeautifulSoup(
                response.content,
                "html.parser"
            )

            articles = soup.find_all("article")

            page_links = []

            for article in articles:

                a_tag = article.find("a")

                if a_tag and a_tag.get("href"):

                    href = a_tag["href"]

                    if "uet.vnu.edu.vn" in href:
                        page_links.append(href)

            post_links.extend(page_links)

            time.sleep(0.5)

        except Exception as e:
            print(f"Error scanning page {page}: {e}")

    return list(set(post_links))

# =========================================================
# STEP 2: CRAWL ARTICLE
# =========================================================

def crawl_url(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(
            response.content,
            "html.parser"
        )

        title_tag = (
            soup.find("h1")
            or soup.find("title")
        )

        title = (
            title_tag.get_text().strip()
            if title_tag
            else "Không có tiêu đề"
        )

        content_div = (
            soup.find("div", class_="entry-content")
            or soup.find("div", class_="post-content")
            or soup.find("article")
        )

        if content_div:

            for tag in content_div(
                ["script", "style", "iframe", "noscript"]
            ):
                tag.decompose()

            content = content_div.get_text(separator="\n")

        else:

            paragraphs = soup.find_all("p")

            content = "\n".join([
                p.get_text().strip()
                for p in paragraphs
                if p.get_text().strip()
            ])

        return {
            "title": title,
            "url": url,
            "raw_content": content
        }

    except Exception as e:
        print(f"Error crawling {url}: {e}")
        return None

# =========================================================
# STEP 3: RULE CLEANING
# =========================================================

def clean_rule_based(text):

    if not text:
        return ""

    text = re.sub(r"<[^>]+>", "", text)

    patterns_to_remove = [

        r"chia sẻ",
        r"facebook",
        r"twitter",
        r"email",
        r"hotline",
        r"xem thêm",
        r"bài viết liên quan",
        r"copyright",
        r"all rights reserved",
        r"menu",
        r"navigation",
        r"đăng nhập",
        r"tìm kiếm"

    ]

    lines = text.split("\n")

    cleaned_lines = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        if len(line) < 5:
            continue

        if any(
            re.search(p, line, re.IGNORECASE)
            for p in patterns_to_remove
        ):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

# =========================================================
# STEP 4: QA GENERATION
# =========================================================

def generate_qa_pairs(docs, batch_size=BATCH_SIZE_QA):

    prompts = []

    for doc in docs:

        text = doc["rule_cleaned_content"][:MAX_INPUT_CHARS]

        messages = [
            {
                "role": "system",
                "content": (
                    "Bạn là chuyên gia thiết kế bộ dữ liệu Hỏi-Đáp (QA) tiếng Việt chất lượng cao để đánh giá hệ thống RAG.\n"
                    "Hãy đọc văn bản được cung cấp và tạo ra CHÍNH XÁC 2 cặp Câu hỏi (CH) và Câu trả lời (ĐA).\n\n"

                    "QUY TẮC BẮT BUỘC ĐỂ CÂU HỎI KHÔNG BỊ MƠ HỒ (QUAN TRỌNG):\n"
                    "1. Câu hỏi PHẢI cực kỳ chi tiết và tự chứa đầy đủ ngữ cảnh (Self-contained). Người đọc không cần đọc văn bản gốc vẫn phải hiểu chính xác câu hỏi đang nói về sự kiện, đối tượng, hay chính sách nào.\n"
                    "2. BẮT BUỘC phải đưa các thông tin sau vào câu hỏi (nếu văn bản có đề cập):\n"
                    "   - Tên cụ thể của chương trình/học bổng/quyết định (Ví dụ: Thay vì hỏi 'Học bổng trị giá bao nhiêu?', hãy hỏi 'Học bổng Mitsubishi năm 2023 trị giá bao nhiêu?').\n"
                    "   - Mốc thời gian cụ thể (Học kỳ, năm học, năm tuyển dụng).\n"
                    "   - Tên đơn vị áp dụng (Trường Đại học Công nghệ hoặc UET).\n"
                    "3. Đáp án phải ngắn gọn (dưới 15 từ), trích xuất chính xác từ văn bản.\n"
                    "4. KHÔNG tạo câu hỏi Yes/No hoặc câu hỏi quá chung chung.\n"
                    "5. Định dạng đầu ra nghiêm ngặt (KHÔNG viết thêm bất kỳ từ nào ngoài định dạng này):\n\n"
                    "CH1: ...\n"
                    "ĐA1: ...\n"
                    "CH2: ...\n"
                    "ĐA2: ..."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Tiêu đề bài viết: {doc['title']}\n\n"
                    f"Văn bản:\n{text}"
                )
            }

        ]

        prompt = qa_pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    # Generator giúp batching tiết kiệm VRAM
    def prompt_generator():
        for p in prompts:
            yield p

    outputs = qa_pipe(
        prompt_generator(),
        batch_size=batch_size,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        return_full_text=False
    )

    results = []

    for out in outputs:

        generated = out[0]["generated_text"]

        if isinstance(generated, list):
            content = generated[-1]["content"]
        else:
            content = generated

        results.append(content.strip())

    return results

# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":

    categories_to_crawl = [

        "https://uet.vnu.edu.vn/doi-song-sinh-vien/",

        "https://uet.vnu.edu.vn/category/tin-tuc/tin-sinh-vien/",
        "https://uet.vnu.edu.vn/tuyen-sinh/",

        "https://uet.vnu.edu.vn/category/tin-tuc/tin-dao-tao/",

        "https://uet.vnu.edu.vn/category/tin-tuc/tin-tuyen-dung/"

    ]

    # =====================================================
    # STEP 1: GET LINKS
    # =====================================================

    print("\n====================")
    print("GETTING LINKS")
    print("====================")

    all_target_urls = []

    for category_url in categories_to_crawl:

        links = get_links_from_category(
            category_url,
            max_pages=MAX_PAGES_PER_CATEGORY
        )

        all_target_urls.extend(links)

    # Remove duplicate
    all_target_urls = list(set(all_target_urls))

    # Shuffle để lấy ngẫu nhiên
    random.shuffle(all_target_urls)

    # Giới hạn khoảng 300 bài
    all_target_urls = all_target_urls[:TARGET_NUM_ARTICLES]

    print(f"\nTotal selected URLs: {len(all_target_urls)}")

    # =====================================================
    # STEP 2: CRAWL CONTENT
    # =====================================================

    print("\n====================")
    print("CRAWLING CONTENT")
    print("====================")

    processed_data = []

    for url in tqdm(all_target_urls):

        doc = crawl_url(url)

        if doc:
            processed_data.append(doc)

    print(f"\nSuccessfully crawled: {len(processed_data)}")

    # =====================================================
    # STEP 3: RULE CLEANING
    # =====================================================

    print("\n====================")
    print("RULE CLEANING")
    print("====================")

    for doc in tqdm(processed_data):

        doc["rule_cleaned_content"] = clean_rule_based(
            doc["raw_content"]
        )

    # =====================================================
    # STEP 4: FILTER LOW QUALITY ARTICLES
    # =====================================================

    processed_data = [

        doc for doc in processed_data

        if len(doc["rule_cleaned_content"]) > 500

    ]

    print(f"\nAfter filtering short articles: {len(processed_data)}")

    # =====================================================
    # STEP 5: QA GENERATION
    # =====================================================

    print("\n====================")
    print("QA GENERATION")
    print("====================")

    qa_results = generate_qa_pairs(
        processed_data,
        batch_size=BATCH_SIZE_QA
    )

    for i, qa in enumerate(qa_results):
        processed_data[i]["generated_qa"] = qa

    # =====================================================
    # STEP 6: SAVE JSON
    # =====================================================

    print("\n====================")
    print("SAVING JSON")
    print("====================")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

        json.dump(
            processed_data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Saved JSON to: {OUTPUT_FILE}")

    # =====================================================
    # STEP 7: EXPORT CSV
    # =====================================================

    print("\n====================")
    print("EXPORTING CSV")
    print("====================")

    try:

        df_qa = parse_qa_to_dataframe(processed_data)

        csv_file = OUTPUT_FILE.replace(".json", ".csv")

        df_qa.to_csv(
            csv_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"Saved CSV to: {csv_file}")

        print(f"Total QA pairs: {len(df_qa)}")

    except Exception as e:

        print(f"Error exporting CSV: {e}")

    # =====================================================
    # PREVIEW
    # =====================================================

    print("\n====================")
    print("SAMPLE OUTPUT")
    print("====================")

    for idx, doc in enumerate(processed_data[:3]):

        print("\n" + "=" * 100)

        print(f"TITLE: {doc['title']}")

        print(f"URL: {doc['url']}")

        print("\n--- GENERATED QA ---")

        print(doc["generated_qa"])


LOADING MODEL


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...


Successfully crawled: 212

RULE CLEANING



100%|██████████| 212/212 [00:00<00:00, 1107.45it/s]



After filtering short articles: 200

QA GENERATION


Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/


SAVING JSON
Saved JSON to: uet_qa_dataset.json

EXPORTING CSV
Saved CSV to: uet_qa_dataset.csv
Total QA pairs: 399

SAMPLE OUTPUT

TITLE: Kế hoạch đánh giá kết quả rèn luyện HKI năm học 2024-2025 và tổ chức Hội nghị đối thoại giữa lãnh đạo Nhà trường với đại biểu sinh viên
URL: https://uet.vnu.edu.vn/ke-hoach-danh-gia-ket-qua-ren-luyen-hki-nam-hoc-2024-2025-va-chuc-hoi-nghi-doi-thoai-giua-lanh-dao-nha-truong-voi-dai-bieu-sinh-vien/

--- GENERATED QA ---
CH1: Khi nào sinh viên thực hiện việc tự đánh giá kết quả rèn luyện học kỳ I năm học 2024-2025?
ĐA1: Từ 8h00 ngày 03/03/2025 đến 8h00 ngày 07/03/2025.

CH2: Link truy cập để đánh giá kết quả rèn luyện học kỳ I năm học 2024-2025 là gì?
ĐA2: https://student.uet.vnu.edu.vn

TITLE: Miễn, giảm học phí và kinh phí được hưởng theo chế độ, trong học kỳ II năm học 2025-2026 của sinh viên đại học chính quy Trường Đại học Công nghệ
URL: https://uet.vnu.edu.vn/mien-giam-hoc-phi-va-kinh-phi-duoc-huong-theo-che-do-trong-hoc-ky-ii-nam-hoc-2025-2026-c

# Create Random Subset

In [2]:
import pandas as pd

# Đường dẫn file gốc
input_path = "/kaggle/working/uet_qa_dataset.csv"

# Đọc file CSV
df = pd.read_csv(input_path)

# =========================
# BƯỚC 1: Gắn ID cho toàn bộ dataset gốc
# =========================
df.insert(0, "ID", range(1, len(df) + 1))

# Lưu file full có ID
full_output_path = "/kaggle/working/uet_qa_dataset_with_id.csv"
df.to_csv(full_output_path, index=False, encoding="utf-8-sig")

print(f"Đã lưu full dataset có ID tại: {full_output_path}")

# =========================
# BƯỚC 2: Random subset khoảng 200 mẫu
# =========================
subset_df = df.sample(n=200, random_state=42).sort_values("ID").reset_index(drop=True)

# Chỉ giữ các cột cần thiết
subset_df = subset_df[["ID", "Title", "URL", "Question", "Answer"]]

# =========================
# BƯỚC 3: Xuất TXT dễ copy sang Google Docs
# =========================
output_txt = "/kaggle/working/uet_qa_subset_200.txt"

with open(output_txt, "w", encoding="utf-8") as f:
    for _, row in subset_df.iterrows():

        f.write("ID\n")
        f.write(f"{row['ID']}\n\n")

        f.write("Title\n")
        f.write(f"{row['Title']}\n\n")

        f.write("URL\n")
        f.write(f"{row['URL']}\n\n")

        f.write("Question\n")
        f.write(f"{row['Question']}\n\n")

        f.write("Answer\n")
        f.write(f"{row['Answer']}\n\n")

        f.write("=" * 80 + "\n\n")

print(f"Đã lưu TXT subset tại: {output_txt}")

# =========================
# BƯỚC 4: Lưu subset CSV
# =========================
output_csv = "/kaggle/working/uet_qa_subset_200.csv"
subset_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Đã lưu subset CSV tại: {output_csv}")

# Preview
print(subset_df.head())

Đã lưu full dataset có ID tại: /kaggle/working/uet_qa_dataset_with_id.csv
Đã lưu TXT subset tại: /kaggle/working/uet_qa_subset_200.txt
Đã lưu subset CSV tại: /kaggle/working/uet_qa_subset_200.csv
   ID                                              Title  \
0   1  Kế hoạch đánh giá kết quả rèn luyện HKI năm họ...   
1   4  Miễn, giảm học phí và kinh phí được hưởng theo...   
2   6  Triển khai tháng hành động Quốc gia phòng, chố...   
3   7  07 sinh viên Trường Đại học Công nghệ nhận học...   
4   8  07 sinh viên Trường Đại học Công nghệ nhận học...   

                                                 URL  \
0  https://uet.vnu.edu.vn/ke-hoach-danh-gia-ket-q...   
1  https://uet.vnu.edu.vn/mien-giam-hoc-phi-va-ki...   
2  https://uet.vnu.edu.vn/trien-khai-thang-hanh-d...   
3  https://uet.vnu.edu.vn/07-sinh-vien-truong-dai...   
4  https://uet.vnu.edu.vn/07-sinh-vien-truong-dai...   

                                            Question  \
0  Khi nào sinh viên thực hiện việc tự đánh giá k.